# Industry Project: An End-to-End MLOps Pipeline on One Machine (Production-Style)

> **Who is this notebook for?** Someone who has trained a model in a notebook and now asks
> *"what happens next in a real company?"* You do not need any MLOps experience. Every step is explained in plain
> English, every function is small, and almost every line has a comment. Everything runs locally, offline, on CPU.

---

### How this notebook is different from the other `15_MLOps` notebooks

| Existing notebooks | This notebook |
|---|---|
| `01_experiment_tracking/01_mlflow.ipynb` shows MLflow logging calls one at a time | Uses tracking as **one step of a pipeline**: nested runs, datasets, artifacts, run comparison |
| `05_model_registry/01_model_registry.ipynb` lists registry APIs (stages, aliases, BentoML, W&B) | Uses **aliases** (`champion` / `challenger` / `previous_champion`) in an **automated promotion decision** |
| `06_monitoring/01_monitoring.ipynb` introduces Evidently / Prometheus | Computes drift and live performance **by hand** and lets the result **trigger retraining** |
| `03_pipeline_orchestration` shows Prefect / Airflow syntax | Builds the pipeline as plain Python functions first, then shows how they map to Airflow / Prefect tasks |
| Each tool on toy data, one at a time | **One complete loop** over several simulated months, including a bad data batch, a bad model and real drift |

---

### The business problem

You work for a parcel delivery company. Every month about 3,000 orders go through one regional hub, and roughly
a third of them arrive **late**. A late parcel costs on average **$12** (support calls, vouchers, churn). If the
operations team warns the customer *before* the parcel is late, the cost drops to about **$4**. The team can contact
**20% of the orders** each day.

A data scientist already showed in a notebook that a model can rank orders by "risk of being late". **Your job is
to make that model run reliably every month without a human babysitting it**:

- new data arrives every month and must be **checked** before anyone trains on it,
- every training run must be **recorded** (who, when, which data, which parameters, which results),
- a new model may only replace the production model if it is **measurably better and passes tests**,
- production predictions must say **which model version** produced them,
- the pipeline must **notice when the world changes** (a new carrier, storm season) and retrain itself.

**Success means:** the business metric (share of late parcels caught within the 20% contact budget, and dollars
saved) stays high month after month, and every production decision is traceable and reversible.

---

### The loop we will build

```
          ┌──────────────────────────────────────────────────────────────────────────┐
          │                                                                          │
 new data ─► 1. validate ─► 2. train + track ─► 3. compare ─► 4. register (challenger) │
                 │                (MLflow)                          │                 │
             stop if bad                                     5. quality gate          │
                                                             pass │      │ fail       │
                                                      promote "champion"  keep old    │
                                                                  │                   │
                         6. batch inference with  models:/...@champion                │
                                                                  │                   │
                         7. monitoring: drift + live performance ─┴── retrain? ───────┘
```

### Table of Contents
1. Setup: config, logging and a private MLflow tracking store
2. The data: monthly delivery batches and a small "data lake"
3. Step 1: data validation (stop bad data early)
4. Step 2: training with experiment tracking (MLflow)
5. Step 3: comparing runs
6. Step 4: registering the best model as the *challenger*
7. Step 5: the automated quality gate and promotion with aliases
8. Step 6: batch inference with the champion
9. Step 7: monitoring, drift and the retraining trigger
10. Step 8: orchestration, one function for the whole loop
11. Registry history, lineage and rollback
12. From this notebook to Airflow / Prefect
13. Automated tests for the pipeline
14. Summary, exercises and resources

---
## 1. Setup: config, logging and a private MLflow tracking store

**MLflow** is the most widely used open-source tool for experiment tracking and model registry. It stores two things:
- the **tracking store / backend**: run metadata (parameters, metrics, tags, registry) in a database,
- the **artifact store**: files (models, plots, reports).

In a company both live on a shared server (`mlflow server --backend-store-uri postgresql://... --artifacts-destination s3://...`).
Here we point MLflow at a **SQLite file and an artifact folder inside a temporary directory**, so the notebook works
offline and never writes into the repository. We set the artifact location explicitly: otherwise MLflow would create
an `mlruns/` folder in the current working directory.

All business numbers and thresholds live in one frozen `Config`, so changing a rule means changing one line.

In [1]:
# ---- Standard library ----
import hashlib                       # fingerprint of the training data (lineage)
import json
import logging
import os
import tempfile                      # private folders for MLflow, the data lake and predictions
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")   # no usage telemetry from a tutorial
os.environ.setdefault("DO_NOT_TRACK", "true")
warnings.filterwarnings("ignore")                          # before the imports: keeps library warnings quiet

# ---- Third-party ----
import matplotlib
matplotlib.use("Agg")                # plots are saved as MLflow artifacts, not shown
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from mlflow.models import infer_signature
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
logging.getLogger("mlflow").setLevel(logging.WARNING)     # keep MLflow's own messages quiet
logging.getLogger("alembic").setLevel(logging.WARNING)
log = logging.getLogger("pipeline")


@dataclass(frozen=True)
class Config:
    seed: int = 42
    model_name: str = "late-delivery-risk"          # name in the model registry
    experiment: str = "late-delivery-risk"          # name of the MLflow experiment
    rows_per_month: int = 3000
    train_window_months: int = 3                    # retrain on the last 3 labelled months
    holdout_days: int = 14                          # the newest 14 days are the time-based holdout
    contact_budget: float = 0.20                    # ops can contact the riskiest 20% of orders
    cost_late: float = 12.0                         # $ per late parcel without warning
    cost_late_warned: float = 4.0                   # $ per late parcel when the customer was warned
    # Quality gate
    min_auc: float = 0.65                           # absolute floor
    min_auc_gain: float = 0.005                     # a challenger must beat the champion by this much
    max_recall_loss: float = 0.01                   # ... and must not lose more business recall than this
    max_ms_per_1k_rows: float = 2000.0              # latency budget for batch scoring (generous: shared CPU)
    # Monitoring
    psi_alert: float = 0.25                         # PSI above this = major distribution shift
    max_auc_drop: float = 0.03                      # live AUC this much below the holdout AUC = degraded
    max_calibration_gap: float = 0.05               # |mean predicted risk - actual late rate| above this = stale


CFG = Config()

WORK_DIR = Path(tempfile.mkdtemp(prefix="mlops_pipeline_"))   # real life: servers, S3 buckets, a warehouse
LAKE_DIR = WORK_DIR / "lake"                                    # labelled monthly data ("feature store")
PRED_DIR = WORK_DIR / "predictions"                             # batch predictions written by the pipeline
for folder in (LAKE_DIR, PRED_DIR):
    folder.mkdir()

mlflow.set_tracking_uri(f"sqlite:///{WORK_DIR / 'mlflow.db'}")  # real life: "http://mlflow.mycompany.com"
if mlflow.get_experiment_by_name(CFG.experiment) is None:
    mlflow.create_experiment(CFG.experiment, artifact_location=(WORK_DIR / "artifacts").as_uri())
mlflow.set_experiment(CFG.experiment)
client = MlflowClient()                                          # lower-level API, used for the registry

log.info("mlflow %s | scikit-learn %s | working dir %s", mlflow.__version__, sklearn.__version__, WORK_DIR)

2026-10-21 18:31:24,829 | INFO | mlflow 3.14.0 | scikit-learn 1.9.0 | working dir /tmp/mlops_pipeline_ts_2qoxa


---
## 2. The data: monthly delivery batches and a small "data lake"

Each month we receive the orders of that month. The label `is_late` is only known **after** delivery, so the labels
for month *M* arrive at the start of month *M+1*. That delay matters for monitoring (Section 9).

The simulator produces realistic data: the chance of being late depends on distance, weather, carrier, number of
items, promised delivery time, weekend and evening orders, plus plenty of unexplained noise (no model can be
perfect). It also contains mess: the weather API is down for ~3% of orders (missing values).

And it contains **a change in the world**: from **May 2026** a new, cheaper carrier `SpeedyX` takes 25% of the
orders (and is often late), and storm season makes the weather worse. The model trained in spring has never seen
this. Will the pipeline notice?

In [2]:
CARRIER_EFFECT = {"FastShip": -0.4, "EcoPost": 0.35, "CityCourier": 0.0, "SpeedyX": 2.5}   # hidden truth
NUMERIC = ["distance_km", "n_items", "promised_days", "weather_severity", "order_hour", "is_weekend"]
CATEGORICAL = ["carrier", "warehouse"]
FEATURES = NUMERIC + CATEGORICAL
TARGET = "is_late"


def simulate_month(month: str, cfg: Config = CFG) -> pd.DataFrame:
    """All orders of one month with their (later known) label. Same month -> same data (seeded)."""
    period = pd.Period(month, freq="M")
    rng = np.random.default_rng(cfg.seed + period.ordinal)            # deterministic per month
    n = cfg.rows_per_month
    new_world = month >= "2026-05"                                      # SpeedyX + storm season
    carriers = ["FastShip", "EcoPost", "CityCourier"] + (["SpeedyX"] if new_world else [])
    carrier_p = [0.30, 0.25, 0.20, 0.25] if new_world else [0.40, 0.35, 0.25]

    df = pd.DataFrame({
        "order_id": [f"{period.strftime('%Y%m')}-{i:05d}" for i in range(n)],
        "order_ts": period.start_time + pd.to_timedelta(rng.uniform(0, period.days_in_month, n), unit="D"),
        "carrier": rng.choice(carriers, n, p=carrier_p),
        "warehouse": rng.choice(["north", "south", "east"], n, p=[0.40, 0.35, 0.25]),
        "distance_km": rng.gamma(2.0, 90, n).round(1),                  # most parcels travel 50-300 km
        "n_items": rng.poisson(1.5, n) + 1,
        "promised_days": rng.choice([1, 2, 3, 5], n, p=[0.2, 0.4, 0.3, 0.1]),
        "weather_severity": (rng.beta(4, 6, n) if new_world else rng.beta(2, 5, n)).round(3),  # 0 = calm, 1 = storm
    })
    df["order_hour"] = df["order_ts"].dt.hour
    df["is_weekend"] = (df["order_ts"].dt.dayofweek >= 5).astype(int)

    logit = (-1.0 + 0.004 * (df["distance_km"] - 180) + 2.0 * (df["weather_severity"] - 0.3)
             + df["carrier"].map(CARRIER_EFFECT) + 0.15 * (df["n_items"] - 2) + 0.3 * df["is_weekend"]
             - 0.35 * (df["promised_days"] - 2) + 0.4 * (df["order_hour"] >= 20)
             + rng.normal(0, 0.9, n))                                    # what no feature can explain
    df[TARGET] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    df.loc[rng.random(n) < 0.03, "weather_severity"] = np.nan          # weather API outages
    return df.sort_values("order_ts", ignore_index=True)


def ingest(month: str, labelled: bool) -> pd.DataFrame:
    """What the pipeline receives: this month's open orders (no label yet) or last month's labelled orders."""
    df = simulate_month(month)
    return df if labelled else df.drop(columns=[TARGET])


def save_to_lake(month: str, df: pd.DataFrame) -> None:
    df.to_parquet(LAKE_DIR / f"{month}.parquet", index=False)        # real life: s3://lake/deliveries/month=.../


def read_lake(months: list[str]) -> pd.DataFrame:
    return pd.concat([pd.read_parquet(LAKE_DIR / f"{m}.parquet") for m in months], ignore_index=True)


preview = simulate_month("2026-01")
print(f"{len(preview):,} orders | late rate {preview[TARGET].mean():.1%} | "
      f"missing weather {preview['weather_severity'].isna().mean():.1%}")
preview.head()

3,000 orders | late rate 31.1% | missing weather 2.6%


,order_id,order_ts,carrier,warehouse,distance_km,n_items,promised_days,weather_severity,order_hour,is_weekend,is_late
0,202601-01147,2026-01-01 00:03:46.138309678,EcoPost,north,21.5,2,1,0.375,0,0,1
1,202601-02252,2026-01-01 00:05:37.099198927,CityCourier,east,60.9,4,3,0.339,0,0,0
2,202601-02603,2026-01-01 00:24:13.973545874,EcoPost,south,153.0,2,2,0.106,0,0,0
3,202601-00541,2026-01-01 00:31:25.130598384,FastShip,north,166.8,2,2,0.139,0,0,1
4,202601-00120,2026-01-01 00:45:00.666764464,FastShip,east,396.6,2,2,0.538,0,0,0


---
## 3. Step 1: data validation (stop bad data early)

> *"Garbage in, garbage out"* is the most common cause of ML incidents. Google's *Rules of ML* (Rule #9):
> **detect problems before exporting models**.

The validation step checks the incoming batch against a **data contract** and returns a report with two kinds of
findings:
- **errors** (missing columns, impossible values, too many NULLs, duplicate IDs) stop the pipeline;
- **warnings** (for example a *new category* such as an unknown carrier) let the pipeline continue but are logged,
  because they are often real business changes that monitoring should look at.

Libraries such as `pandera`, Great Expectations or TFDV do the same thing at scale (see `14_data_contracts.ipynb`).

In [3]:
class DataValidationError(Exception):
    """Raised when a batch breaks the data contract. The pipeline stops; nobody trains on bad data."""


CONTRACT = {
    "required_columns": ["order_id", "order_ts"] + FEATURES,
    "ranges": {"distance_km": (0, 3000), "n_items": (1, 100), "promised_days": (1, 10),
               "weather_severity": (0, 1), "order_hour": (0, 23), "is_weekend": (0, 1)},
    "max_null_rate": {"weather_severity": 0.10},                      # other columns: no NULLs allowed
    "known_categories": {"carrier": {"FastShip", "EcoPost", "CityCourier"},
                         "warehouse": {"north", "south", "east"}},
    "min_rows": 500,
}


def validate_batch(df: pd.DataFrame, labelled: bool) -> dict:
    """Check a batch against CONTRACT. Returns {'errors': [...], 'warnings': [...], 'n_rows': ...}."""
    errors, warnings_ = [], []
    required = CONTRACT["required_columns"] + ([TARGET] if labelled else [])
    missing = [c for c in required if c not in df.columns]
    if missing:                                                        # nothing else makes sense without columns
        return {"errors": [f"missing columns: {missing}"], "warnings": [], "n_rows": len(df)}

    if len(df) < CONTRACT["min_rows"]:
        errors.append(f"only {len(df)} rows (expected >= {CONTRACT['min_rows']})")
    if df["order_id"].duplicated().any():
        errors.append(f"{df['order_id'].duplicated().sum()} duplicate order_id values")
    for col in required:
        null_rate = df[col].isna().mean()
        if null_rate > CONTRACT["max_null_rate"].get(col, 0.0):
            errors.append(f"{col}: {null_rate:.1%} NULLs")
    for col, (low, high) in CONTRACT["ranges"].items():
        n_bad = int(((df[col] < low) | (df[col] > high)).sum())       # NaN compares False -> not counted
        if n_bad:
            errors.append(f"{col}: {n_bad} values outside [{low}, {high}]")
    if labelled and not set(df[TARGET].unique()) <= {0, 1}:
        errors.append(f"{TARGET} must be 0/1")
    for col, known in CONTRACT["known_categories"].items():
        new_values = set(df[col].dropna().unique()) - known
        if new_values:
            share = df[col].isin(new_values).mean()
            warnings_.append(f"{col}: new categories {sorted(new_values)} ({share:.0%} of rows)")
    return {"errors": errors, "warnings": warnings_, "n_rows": len(df)}


def require_valid(df: pd.DataFrame, labelled: bool, batch_name: str) -> dict:
    """Validate and raise if there are errors (this is what the pipeline calls)."""
    report = validate_batch(df, labelled)
    for w in report["warnings"]:
        log.warning("[%s] %s", batch_name, w)
    if report["errors"]:
        raise DataValidationError(f"[{batch_name}] " + "; ".join(report["errors"]))
    log.info("[%s] validation passed (%d rows)", batch_name, report["n_rows"])
    return report


# Back-fill the lake with the first two labelled months (validated, of course)
for month in ["2026-01", "2026-02"]:
    batch = ingest(month, labelled=True)
    require_valid(batch, labelled=True, batch_name=month)
    save_to_lake(month, batch)

2026-10-21 18:31:24,964 | INFO | [2026-01] validation passed (3000 rows)


2026-10-21 18:31:25,030 | INFO | [2026-02] validation passed (3000 rows)


What does a **bad batch** look like? Below, an upstream team changed their export: distances are suddenly sent in
*metres*, `order_hour` was renamed, and a bug duplicated rows. Validation stops the pipeline with a clear message,
before any model is trained on it.

In [4]:
broken = ingest("2026-03", labelled=True)
broken["distance_km"] = broken["distance_km"] * 1000                  # unit change upstream (km -> m)
broken = pd.concat([broken, broken.head(50)]).rename(columns={"order_hour": "hour_of_order"})

try:
    require_valid(broken, labelled=True, batch_name="2026-03 (broken export)")
except DataValidationError as err:
    print("PIPELINE STOPPED ->", err)

broken = broken.rename(columns={"hour_of_order": "order_hour"})     # after the upstream team fixes the column name...
print("\nRemaining problems after the rename is fixed:")
for problem in validate_batch(broken, labelled=True)["errors"]:
    print("  -", problem)

PIPELINE STOPPED -> [2026-03 (broken export)] missing columns: ['order_hour']

Remaining problems after the rename is fixed:
  - 50 duplicate order_id values
  - distance_km: 3048 values outside [0, 3000]


---
## 4. Step 2: training with experiment tracking (MLflow)

Every training attempt becomes an MLflow **run**. We log:

| What | MLflow call | Why |
|---|---|---|
| hyper-parameters | `log_params` | reproduce the run |
| metrics (technical and business) | `log_metrics` | compare runs |
| the training data (name, digest, schema) | `log_input(mlflow.data.from_pandas(...))` | data lineage |
| tags (data window, data hash, code version) | `set_tags` | search and audit |
| files (plots, reports) | `log_figure`, `log_dict` | humans can inspect a run |
| the model itself + its input/output **signature** | `mlflow.sklearn.log_model` | load it later anywhere |

The whole monthly pipeline is one **parent run**; each candidate model is a **nested child run** under it, so the
UI shows "pipeline run of 2026-04-01 -> 3 candidates".

The split is **time-based**: we train on older orders and evaluate on the newest 14 days (the *holdout*), because in
production we always predict the future. The business metric is **recall within the contact budget**: of all late
parcels in the holdout, what share is in the riskiest 20% the model would flag?

In [5]:
def recall_at_budget(y_true, scores, budget: float = CFG.contact_budget) -> float:
    """Share of all late parcels that are inside the top-`budget` fraction of risk scores."""
    y_true = np.asarray(y_true)
    n_flag = int(np.ceil(budget * len(scores)))
    flagged = np.argsort(-np.asarray(scores))[:n_flag]                 # indices of the riskiest orders
    return float(y_true[flagged].sum() / max(y_true.sum(), 1))


def evaluate(model, df: pd.DataFrame) -> dict:
    """Technical + business metrics on a labelled DataFrame."""
    scores = model.predict_proba(df[FEATURES])[:, 1]
    recall = recall_at_budget(df[TARGET], scores)
    caught = recall * df[TARGET].sum()                                  # late parcels we warn about
    return {"roc_auc": roc_auc_score(df[TARGET], scores),
            "pr_auc": average_precision_score(df[TARGET], scores),
            "recall_at_budget": recall,
            "usd_saved_per_month": caught * (CFG.cost_late - CFG.cost_late_warned)
                                   * (CFG.rows_per_month / len(df))}     # scale holdout to a full month


def make_pipeline(estimator, numeric=NUMERIC, categorical=CATEGORICAL) -> Pipeline:
    """Preprocessing + model in ONE object, so serving does exactly what training did."""
    preprocess = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),   # unseen carrier -> all zeros, no crash
    ])                                                                  # columns not listed are dropped
    return Pipeline([("preprocess", preprocess), ("model", estimator)])


CANDIDATES = {                                                           # name -> (estimator, params to log)
    "baseline_prior": (DummyClassifier(strategy="prior"), {"strategy": "prior"}),
    "logistic_regression": (LogisticRegression(C=1.0, max_iter=1000), {"C": 1.0}),
    "gradient_boosting": (HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=4,
                                                         random_state=CFG.seed),
                          {"max_iter": 200, "learning_rate": 0.05, "max_depth": 4}),
}


def data_fingerprint(df: pd.DataFrame) -> str:
    """Short hash of the exact training rows: same hash = same data (lineage, reproducibility)."""
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=False).values.tobytes()).hexdigest()[:12]


def time_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Train on everything except the newest `holdout_days`; those days are the holdout."""
    cutoff = df["order_ts"].max() - pd.Timedelta(days=CFG.holdout_days)
    return df[df["order_ts"] <= cutoff], df[df["order_ts"] > cutoff]

In [6]:
def plot_scores(scores, y_true, title: str):
    """A small diagnostic plot that is stored with the run (humans like pictures)."""
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(scores[y_true == 0], bins=30, alpha=0.6, label="on time", density=True)
    ax.hist(scores[y_true == 1], bins=30, alpha=0.6, label="late", density=True)
    ax.set_xlabel("predicted risk of being late"); ax.set_title(title); ax.legend()
    fig.tight_layout()
    return fig


def train_candidates(train: pd.DataFrame, holdout: pd.DataFrame, data_window: str, note: str = "",
                     candidates: dict = CANDIDATES, numeric=NUMERIC, categorical=CATEGORICAL) -> pd.DataFrame:
    """Train every candidate as a nested MLflow run. Returns one row per candidate with its metrics."""
    rows = []
    feature_list = list(numeric) + list(categorical)                    # the columns the model really uses
    for name, (estimator, params) in candidates.items():
        with mlflow.start_run(run_name=name, nested=True) as run:
            model = make_pipeline(clone(estimator), numeric, categorical)  # fresh, unfitted copy each time
            model.fit(train[FEATURES], train[TARGET])
            metrics = evaluate(model, holdout)

            mlflow.log_params({**params, "algorithm": name, "n_features": len(feature_list)})
            mlflow.log_metrics({f"holdout_{k}": v for k, v in metrics.items()})
            mlflow.set_tags({"run_type": "candidate", "data_window": data_window, "note": note,
                             "train_data_hash": data_fingerprint(train[feature_list + [TARGET]]),
                             "git_commit": os.environ.get("GITHUB_SHA", "local-notebook")})  # CI sets GITHUB_SHA
            mlflow.log_input(mlflow.data.from_pandas(train[feature_list + [TARGET]], targets=TARGET,
                                                     name=f"deliveries_{data_window}"), context="training")
            scores = model.predict_proba(holdout[FEATURES])[:, 1]
            mlflow.log_figure(plot_scores(scores, holdout[TARGET].values, name), "plots/score_distribution.png")
            plt.close("all")

            model_uri = None
            if name != "baseline_prior":                                  # the dummy is only a yardstick
                model_info = mlflow.sklearn.log_model(
                    model, name="model",
                    signature=infer_signature(holdout[FEATURES].head(100), scores[:100]),  # input/output schema
                    pip_requirements=[f"scikit-learn=={sklearn.__version__}", f"pandas=={pd.__version__}"],
                    # MLflow 3 saves sklearn models with `skops` (safer than pickle: loading cannot run arbitrary
                    # code). Types skops does not know must be explicitly trusted:
                    skops_trusted_types=["numpy.dtype"],
                )
                model_uri = model_info.model_uri                          # MLflow 3: "models:/m-<model id>"
            rows.append({"candidate": name, "run_id": run.info.run_id, "model_uri": model_uri, **metrics})
            log.info("  %-20s holdout ROC-AUC %.3f | recall@20%% %.3f", name, metrics["roc_auc"],
                     metrics["recall_at_budget"])
    return pd.DataFrame(rows)

Now we run the **first pipeline execution by hand** (date: 2026-04-01). March's labels have just arrived: validate
them, add them to the lake, train on January-March, evaluate on the last 14 days of March.

In [7]:
RUN_DATE = "2026-04-01"
march = ingest("2026-03", labelled=True)
require_valid(march, labelled=True, batch_name="2026-03")
save_to_lake("2026-03", march)

window_months = ["2026-01", "2026-02", "2026-03"]
train_df, holdout_df = time_split(read_lake(window_months))
print(f"train {len(train_df):,} rows ({train_df['order_ts'].min():%Y-%m-%d} .. {train_df['order_ts'].max():%Y-%m-%d}) | "
      f"holdout {len(holdout_df):,} rows ({holdout_df['order_ts'].min():%Y-%m-%d} .. {holdout_df['order_ts'].max():%Y-%m-%d})")

t0 = time.perf_counter()
with mlflow.start_run(run_name=f"pipeline-{RUN_DATE}") as parent_run:
    mlflow.set_tags({"run_type": "pipeline", "run_date": RUN_DATE})
    mlflow.log_dict(validate_batch(march, labelled=True), "validation/2026-03.json")   # keep the evidence
    results_v1 = train_candidates(train_df, holdout_df, data_window="2026-01..2026-03")
log.info("training step took %.0f s", time.perf_counter() - t0)
results_v1.drop(columns=["run_id", "model_uri"]).round(3)

2026-10-21 18:31:25,231 | INFO | [2026-03] validation passed (3000 rows)


train 7,622 rows (2026-01-01 .. 2026-03-17) | holdout 1,378 rows (2026-03-18 .. 2026-03-31)


2026-10-21 18:31:26,187 | INFO |   baseline_prior       holdout ROC-AUC 0.500 | recall@20% 0.190


2026-10-21 18:31:37,746 | INFO |   logistic_regression  holdout ROC-AUC 0.671 | recall@20% 0.325


2026-10-21 18:31:39,025 | INFO |   gradient_boosting    holdout ROC-AUC 0.663 | recall@20% 0.325


2026-10-21 18:31:39,061 | INFO | training step took 14 s


,candidate,roc_auc,pr_auc,recall_at_budget,usd_saved_per_month
0,baseline_prior,0.500,0.328,0.190,1497.823
1,logistic_regression,0.671,0.506,0.325,2560.232
2,gradient_boosting,0.663,0.492,0.325,2560.232


---
## 5. Step 3: comparing runs

Everything we logged can now be **queried**. `mlflow.search_runs` returns a pandas DataFrame, which is how
teams build leaderboards, reports and automated decisions. (Interactively you would run
`mlflow ui --backend-store-uri sqlite:///<path>/mlflow.db` and compare runs in the browser.)

In [8]:
leaderboard = mlflow.search_runs(
    experiment_names=[CFG.experiment],
    filter_string=f"tags.run_type = 'candidate' and tags.data_window = '2026-01..2026-03'",
    order_by=["metrics.holdout_roc_auc DESC"],
)
cols = ["tags.mlflow.runName", "params.algorithm", "metrics.holdout_roc_auc", "metrics.holdout_pr_auc",
        "metrics.holdout_recall_at_budget", "metrics.holdout_usd_saved_per_month", "tags.train_data_hash"]
print(leaderboard[cols].round(3).to_string(index=False))

# The parent/child structure is stored too: every candidate points to its pipeline run
print("\nAll candidates belong to pipeline run:", leaderboard["tags.mlflow.parentRunId"].unique())

tags.mlflow.runName    params.algorithm  metrics.holdout_roc_auc  metrics.holdout_pr_auc  metrics.holdout_recall_at_budget  metrics.holdout_usd_saved_per_month tags.train_data_hash
logistic_regression logistic_regression                    0.671                   0.506                             0.325                             2560.232         b52fad9b7d2b
  gradient_boosting   gradient_boosting                    0.663                   0.492                             0.325                             2560.232         b52fad9b7d2b
     baseline_prior      baseline_prior                    0.500                   0.328                             0.190                             1497.823         b52fad9b7d2b

All candidates belong to pipeline run: ['3385c2a2f1094e17963d67cd9e0ff2ac']


How to read the leaderboard:
- The **prior baseline** (always predicts the average late rate) has ROC-AUC 0.5 and catches ~20% of late parcels
  with a 20% budget, exactly what random contacting would do. Any real model must beat it clearly.
- Both real models beat it clearly (ROC-AUC ~0.67, about one third of late parcels caught with the same budget).
  Logistic regression is marginally ahead of gradient boosting, which is a tie in practice: on this mostly additive
  problem the simple model is as good as the complex one, and it is faster and easier to explain.
- The holdout AUC of ~0.67 is modest, and that is realistic: much of what makes a parcel late (traffic, a sick
  driver) is not in the data. A model does not need to be perfect to save money.
- All candidates share the same `train_data_hash`: they were compared on **identical data**, which makes the
  comparison fair.

---
## 6. Step 4: registering the best model as the *challenger*

The **model registry** is the catalogue of models that are candidates for production. Registering creates a new
**version** of the registered model `late-delivery-risk`. Each version points back to the run that produced it
(lineage: version -> run -> parameters, data, code).

Instead of the old "stages" (Staging/Production, deprecated since MLflow 2.9), MLflow now recommends **aliases**:
movable named pointers to a version. We use three:

| Alias | Meaning |
|---|---|
| `challenger` | the newest candidate waiting for the quality gate |
| `champion` | the version production loads (`models:/late-delivery-risk@champion`) |
| `previous_champion` | the version to roll back to |

Because production code loads `@champion`, **promoting or rolling back is just moving a pointer**, no redeploy.

In [9]:
def register_best(results: pd.DataFrame, description: str) -> str:
    """Register the best non-baseline candidate and point the 'challenger' alias at it. Returns the version."""
    best = results[results["candidate"] != "baseline_prior"].sort_values("roc_auc", ascending=False).iloc[0]
    model_version = mlflow.register_model(best["model_uri"], CFG.model_name)   # the logged model's URI
    client.set_registered_model_alias(CFG.model_name, "challenger", model_version.version)
    client.update_model_version(CFG.model_name, model_version.version, description=description)
    client.set_model_version_tag(CFG.model_name, model_version.version, "algorithm", best["candidate"])
    client.set_model_version_tag(CFG.model_name, model_version.version, "holdout_roc_auc", f"{best['roc_auc']:.4f}")
    log.info("registered %s version %s (%s) as 'challenger'", CFG.model_name, model_version.version, best["candidate"])
    return model_version.version


v1 = register_best(results_v1, "First model, trained on 2026-01..2026-03")
mv = client.get_model_version_by_alias(CFG.model_name, "challenger")
print(f"challenger -> version {mv.version} | run {mv.run_id[:8]} | tags {mv.tags}")

Successfully registered model 'late-delivery-risk'.
Created version '1' of model 'late-delivery-risk'.


2026-10-21 18:31:39,347 | INFO | registered late-delivery-risk version 1 (logistic_regression) as 'challenger'


challenger -> version 1 | run f2586251 | tags {'algorithm': 'logistic_regression', 'holdout_roc_auc': '0.6714'}


---
## 7. Step 5: the automated quality gate and promotion with aliases

A model is only promoted if it passes **every** check of the gate. This is the ML version of "all tests must be
green before merging". Our gate combines the three kinds of tests recommended by Made With ML and Google's
*ML Test Score*:

1. **Performance vs. the champion** on the *same* holdout (a fair, head-to-head comparison), plus an absolute floor.
2. **Business metric** must not get worse (a model can win on AUC and lose on the metric the business pays for).
3. **Behavioural tests**: it must not crash on missing weather or an unseen carrier, must return valid
   probabilities, and longer distances must not *lower* the predicted risk on average (a directional expectation).
4. **Operational test**: scoring 1,000 orders must fit the latency budget.

If there is no champion yet (the very first model), it must beat the prior baseline instead.

In [10]:
def get_version_by_alias(alias: str):
    """The model version behind an alias, or None if the alias does not exist yet."""
    try:
        return client.get_model_version_by_alias(CFG.model_name, alias)
    except MlflowException:
        return None


def load_version(version: str):
    return mlflow.sklearn.load_model(f"models:/{CFG.model_name}/{version}")


def quality_gate(challenger_version: str, holdout: pd.DataFrame) -> tuple[bool, pd.DataFrame]:
    """Run all checks. Returns (passed, table of checks)."""
    challenger = load_version(challenger_version)
    champion_mv = get_version_by_alias("champion")
    checks = []

    def check(name, passed, detail):
        checks.append({"check": name, "passed": bool(passed), "detail": detail})

    # 1 + 2: performance, compared head-to-head on the SAME holdout
    new = evaluate(challenger, holdout)
    check("absolute ROC-AUC floor", new["roc_auc"] >= CFG.min_auc, f"{new['roc_auc']:.3f} >= {CFG.min_auc}")
    if champion_mv is None:
        prior = make_pipeline(DummyClassifier(strategy="prior")).fit(holdout[FEATURES], holdout[TARGET])
        ref, ref_name = evaluate(prior, holdout), "prior baseline"
    else:
        ref, ref_name = evaluate(load_version(champion_mv.version), holdout), f"champion v{champion_mv.version}"
    check(f"ROC-AUC beats {ref_name}", new["roc_auc"] >= ref["roc_auc"] + CFG.min_auc_gain,
          f"{new['roc_auc']:.3f} vs {ref['roc_auc']:.3f} (+{CFG.min_auc_gain} required)")
    check(f"recall@budget not worse than {ref_name}",
          new["recall_at_budget"] >= ref["recall_at_budget"] - CFG.max_recall_loss,
          f"{new['recall_at_budget']:.3f} vs {ref['recall_at_budget']:.3f}")

    # 3: behavioural tests
    sample = holdout[FEATURES].head(200).copy()
    weird = sample.assign(weather_severity=np.nan, carrier="BrandNewCarrier")
    p_weird = challenger.predict_proba(weird)[:, 1]
    check("handles missing weather + unseen carrier", np.isfinite(p_weird).all(), "no crash, finite outputs")
    p = challenger.predict_proba(sample)[:, 1]
    check("probabilities in [0, 1]", ((p >= 0) & (p <= 1)).all(), f"min {p.min():.3f}, max {p.max():.3f}")
    p_far = challenger.predict_proba(sample.assign(distance_km=sample["distance_km"] + 500))[:, 1]
    check("directional: +500 km does not lower risk", p_far.mean() >= p.mean(),
          f"mean risk {p.mean():.3f} -> {p_far.mean():.3f}")

    # 4: latency
    batch = pd.concat([holdout[FEATURES]] * (1000 // len(holdout) + 1)).head(1000)
    t0 = time.perf_counter(); challenger.predict_proba(batch); ms = (time.perf_counter() - t0) * 1000
    check("latency for 1,000 rows", ms <= CFG.max_ms_per_1k_rows, f"{ms:.0f} ms (budget {CFG.max_ms_per_1k_rows:.0f})")

    report = pd.DataFrame(checks)
    return bool(report["passed"].all()), report


def promote_or_reject(version: str, passed: bool, report: pd.DataFrame) -> None:
    """Move aliases (promotion) or record the rejection. Either way the decision is stored in the registry."""
    client.set_model_version_tag(CFG.model_name, version, "quality_gate", "passed" if passed else "failed")
    client.set_model_version_tag(CFG.model_name, version, "gate_report", report.to_json(orient="records")[:4900])
    if not passed:
        log.warning("version %s REJECTED by the quality gate; champion unchanged", version)
        return
    old = get_version_by_alias("champion")
    if old is not None:
        client.set_registered_model_alias(CFG.model_name, "previous_champion", old.version)
    client.set_registered_model_alias(CFG.model_name, "champion", version)
    log.info("version %s PROMOTED to champion (previous: %s)", version, old.version if old else "none")


passed, gate_report = quality_gate(v1, holdout_df)
print(gate_report.to_string(index=False))
promote_or_reject(v1, passed, gate_report)

                                      check  passed                           detail
                     absolute ROC-AUC floor    True                    0.671 >= 0.65
               ROC-AUC beats prior baseline    True 0.671 vs 0.500 (+0.005 required)
recall@budget not worse than prior baseline    True                   0.325 vs 0.190
   handles missing weather + unseen carrier    True         no crash, finite outputs
                    probabilities in [0, 1]    True             min 0.053, max 0.783
   directional: +500 km does not lower risk    True         mean risk 0.288 -> 0.641
                     latency for 1,000 rows    True               7 ms (budget 2000)


2026-10-21 18:31:39,596 | INFO | version 1 PROMOTED to champion (previous: none)


The first model passed every check and is now the `champion`.

### What if a teammate's change makes the model worse?
A refactor of the feature code accidentally drops the `weather_severity` and `carrier` columns. The code still runs,
the model still trains, and nobody notices by eye. We train it, register it as a challenger, and let the gate decide.

In [11]:
BUGGY_NUMERIC = [c for c in NUMERIC if c != "weather_severity"]      # the "refactor" bug ...
BUGGY_CATEGORICAL = ["warehouse"]                                      # ... silently dropped 2 columns

with mlflow.start_run(run_name=f"pipeline-{RUN_DATE}-teammate-branch"):
    mlflow.set_tags({"run_type": "pipeline", "run_date": RUN_DATE, "branch": "feature/refactor-features"})
    buggy_results = train_candidates(train_df, holdout_df, "2026-01..2026-03",
                                     note="feature refactor (bug: dropped 2 columns)",
                                     candidates={"gradient_boosting": CANDIDATES["gradient_boosting"]},
                                     numeric=BUGGY_NUMERIC, categorical=BUGGY_CATEGORICAL)

v_buggy = register_best(buggy_results, "Teammate's refactor branch")
passed_buggy, gate_buggy = quality_gate(v_buggy, holdout_df)
print(gate_buggy.to_string(index=False))
promote_or_reject(v_buggy, passed_buggy, gate_buggy)
print("\nchampion is still version", get_version_by_alias("champion").version)

2026-10-21 18:31:40,936 | INFO |   gradient_boosting    holdout ROC-AUC 0.626 | recall@20% 0.292


Registered model 'late-delivery-risk' already exists. Creating a new version of this model...
Created version '2' of model 'late-delivery-risk'.
2026-10-21 18:31:41,137 | INFO | registered late-delivery-risk version 2 (gradient_boosting) as 'challenger'


2026-10-21 18:31:41,518 | WARNING | version 2 REJECTED by the quality gate; champion unchanged


                                   check  passed                           detail
                  absolute ROC-AUC floor   False                    0.626 >= 0.65
               ROC-AUC beats champion v1   False 0.626 vs 0.671 (+0.005 required)
recall@budget not worse than champion v1   False                   0.292 vs 0.325
handles missing weather + unseen carrier    True         no crash, finite outputs
                 probabilities in [0, 1]    True             min 0.062, max 0.807
directional: +500 km does not lower risk    True         mean risk 0.285 -> 0.659
                  latency for 1,000 rows    True              15 ms (budget 2000)

champion is still version 1


The gate **rejected** the buggy model: it falls below the absolute AUC floor and loses to the champion on the same
holdout, both on ROC-AUC and on the business metric. The champion alias did not move. The rejection and the full gate report are stored as tags on the model version, so in the review
the team can see *why* it was blocked. (The behavioural checks also show something subtle: the buggy model does not
crash on an unseen carrier simply because it no longer looks at the carrier at all.)

---
## 8. Step 6: batch inference with the champion

Many ML systems are **batch jobs**: once a day or month, score all open orders and write the results to a table
that other systems read. The scoring job never hard-codes a version; it loads `models:/late-delivery-risk@champion`,
so a promotion takes effect on the next run.

Every prediction row stores the **model version** that produced it. When a customer complains or an auditor asks,
you can trace a single prediction back to the model version, the training run, the data and the code.

In [12]:
def score_batch(month: str) -> pd.DataFrame:
    """Score this month's open orders with the current champion and save the predictions (with lineage)."""
    orders = ingest(month, labelled=False)
    require_valid(orders, labelled=False, batch_name=f"{month} (to score)")
    champion_mv = get_version_by_alias("champion")
    model = mlflow.sklearn.load_model(f"models:/{CFG.model_name}@champion")      # load by ALIAS, not version
    scores = model.predict_proba(orders[FEATURES])[:, 1]
    predictions = pd.DataFrame({
        "order_id": orders["order_id"],
        "risk_of_late": scores.round(4),
        "contact_customer": scores >= np.quantile(scores, 1 - CFG.contact_budget),   # top 20% riskiest
        "model_name": CFG.model_name,
        "model_version": champion_mv.version,                                          # lineage!
        "scored_at": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    })
    predictions.to_parquet(PRED_DIR / f"{month}.parquet", index=False)              # real life: a warehouse table
    log.info("scored %d orders of %s with %s v%s (%d to contact)", len(predictions), month,
             CFG.model_name, champion_mv.version, predictions["contact_customer"].sum())
    return predictions


april_predictions = score_batch("2026-04")
april_predictions.head()

2026-10-21 18:31:41,593 | INFO | [2026-04 (to score)] validation passed (3000 rows)


2026-10-21 18:31:41,653 | INFO | scored 3000 orders of 2026-04 with late-delivery-risk v1 (600 to contact)


,order_id,risk_of_late,contact_customer,model_name,model_version,scored_at
0,202604-02300,0.2006,False,late-delivery-risk,1,2026-10-21T13:31:41+00:00
1,202604-02116,0.1565,False,late-delivery-risk,1,2026-10-21T13:31:41+00:00
2,202604-00389,0.3968,False,late-delivery-risk,1,2026-10-21T13:31:41+00:00
3,202604-02351,0.1708,False,late-delivery-risk,1,2026-10-21T13:31:41+00:00
4,202604-01807,0.1789,False,late-delivery-risk,1,2026-10-21T13:31:41+00:00


---
## 9. Step 7: monitoring, drift and the retraining trigger

Models do not break loudly; they **decay silently** as the world changes (Chip Huyen, *Designing ML Systems*,
chapter 8). We monitor three things every month:

1. **Data drift** (no labels needed): does the new data look like the training data? We use the **Population
   Stability Index (PSI)** per feature: < 0.1 stable, 0.1-0.25 moderate shift, > 0.25 major shift.
2. **Live ranking performance** (needs labels): when last month's labels arrive, we join them with the
   predictions we actually made and compute the real ROC-AUC. If it is clearly below the holdout AUC measured
   before promotion, the model has degraded.
3. **Calibration**: does the average predicted risk still match the real late rate? A model can still *rank*
   orders reasonably while its probabilities have become wrong (and decisions based on them are then wrong too).

The retraining **trigger** is a rule on these numbers. Here: retrain if live ROC-AUC dropped by more than 0.03,
if the predicted and actual late rates differ by more than 5 points, or if any feature has PSI > 0.25.

In [13]:
def psi(reference: pd.Series, current: pd.Series, bins: int = 10) -> float:
    """Population Stability Index for a numeric or categorical feature."""
    reference, current = reference.dropna(), current.dropna()
    if not pd.api.types.is_numeric_dtype(reference):                       # categorical: compare category shares
        cats = sorted(set(reference) | set(current))
        ref_pct = reference.value_counts(normalize=True).reindex(cats, fill_value=0).values
        cur_pct = current.value_counts(normalize=True).reindex(cats, fill_value=0).values
    else:                                                                   # numeric: bins from the reference
        edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
        edges[0], edges[-1] = -np.inf, np.inf
        ref_pct = np.histogram(reference, edges)[0] / len(reference)
        cur_pct = np.histogram(current, edges)[0] / len(current)
    ref_pct, cur_pct = np.clip(ref_pct, 1e-4, None), np.clip(cur_pct, 1e-4, None)   # avoid log(0)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))


def monitor(labelled_month: str) -> dict:
    """Compare a freshly labelled month with the champion's training data and with its own predictions."""
    champion_mv = get_version_by_alias("champion")
    run = client.get_run(champion_mv.run_id)
    first, last = run.data.tags["data_window"].split("..")                   # lineage: which data trained it
    reference = read_lake([str(p) for p in pd.period_range(first, last, freq="M")])
    current = read_lake([labelled_month])

    drift = {f: psi(reference[f], current[f]) for f in ["distance_km", "weather_severity", "n_items", "carrier"]}
    preds = pd.read_parquet(PRED_DIR / f"{labelled_month}.parquet").merge(current[["order_id", TARGET]], on="order_id")
    live_auc = roc_auc_score(preds[TARGET], preds["risk_of_late"])
    live_recall = recall_at_budget(preds[TARGET], preds["risk_of_late"])
    holdout_auc = run.data.metrics["holdout_roc_auc"]

    mean_predicted, late_rate = preds["risk_of_late"].mean(), preds[TARGET].mean()   # calibration check

    reasons = [f"PSI({f})={v:.2f}" for f, v in drift.items() if v > CFG.psi_alert]
    if live_auc < holdout_auc - CFG.max_auc_drop:
        reasons.append(f"live AUC {live_auc:.3f} < holdout AUC {holdout_auc:.3f} - {CFG.max_auc_drop}")
    if abs(mean_predicted - late_rate) > CFG.max_calibration_gap:
        reasons.append(f"predicted late rate {mean_predicted:.1%} vs actual {late_rate:.1%}")
    return {"month": labelled_month, "champion_version": champion_mv.version,
            "holdout_auc": round(holdout_auc, 3), "live_auc": round(live_auc, 3),
            "live_recall_at_budget": round(live_recall, 3),
            "live_mean_predicted": round(mean_predicted, 3), "live_late_rate": round(late_rate, 3),
            **{f"psi_{f}": round(v, 3) for f, v in drift.items()},
            "retrain": bool(reasons), "reasons": "; ".join(reasons) or "all green"}


# The 2026-05-01 run will receive April's labels; we can already peek at what monitoring will say
april = ingest("2026-04", labelled=True)
save_to_lake("2026-04", april)
pd.Series(monitor("2026-04"))

month                      2026-04
champion_version                 1
holdout_auc                  0.671
live_auc                     0.686
live_recall_at_budget        0.336
live_mean_predicted          0.315
live_late_rate               0.314
psi_distance_km              0.005
psi_weather_severity         0.004
psi_n_items                  0.001
psi_carrier                    0.0
retrain                      False
reasons                  all green
dtype: object

April looks healthy: small PSI values and a live ROC-AUC close to the holdout estimate, so **no retraining** is
needed. Retraining when nothing changed costs compute and adds risk for no benefit; "retrain every night just
because" is not a best practice.

---
## 10. Step 8: orchestration, one function for the whole loop

Now we chain every step into **one function** that runs on the first day of each month. This is exactly what an
orchestrator (Airflow, Prefect, Dagster, Kubeflow) would schedule; Section 12 shows the mapping.

Design rules that make pipelines safe to automate:
- **Each step is a function** with clear inputs and outputs (easy to test, easy to turn into a task).
- **Fail fast**: invalid data raises before anything is trained or scored.
- **Idempotent**: re-running the same month overwrites the same outputs instead of duplicating them.
- **Everything is logged** in one parent MLflow run per execution, so you can audit any month later.

In [14]:
def previous_month(run_date: str) -> str:
    return str(pd.Period(run_date, freq="M") - 1)


def run_monthly_pipeline(run_date: str) -> dict:
    """The full loop for one month. run_date is the first day of the month the job runs."""
    labelled_month, score_month = previous_month(run_date), str(pd.Period(run_date, freq="M"))
    summary = {"run_date": run_date}
    with mlflow.start_run(run_name=f"pipeline-{run_date}"):
        mlflow.set_tags({"run_type": "pipeline", "run_date": run_date})

        # 1. Ingest + validate last month's labels, then store them in the lake
        labelled = ingest(labelled_month, labelled=True)
        report = require_valid(labelled, labelled=True, batch_name=labelled_month)
        mlflow.log_dict(report, f"validation/{labelled_month}.json")
        save_to_lake(labelled_month, labelled)

        # 2. Monitoring decides whether we need a new model
        if get_version_by_alias("champion") is None:
            decision = {"retrain": True, "reasons": "no champion yet"}
        else:
            decision = monitor(labelled_month)
            mlflow.log_metrics({k: v for k, v in decision.items() if k.startswith(("psi_", "live_"))})
        mlflow.set_tag("retrain_reason", decision["reasons"])
        summary.update(live_auc=decision.get("live_auc"), monitoring=decision["reasons"])

        # 3-5. Retrain -> register challenger -> quality gate -> promote or reject
        if decision["retrain"]:
            last = pd.Period(labelled_month, freq="M")
            months = [str(last - i) for i in reversed(range(CFG.train_window_months))]
            train, holdout = time_split(read_lake(months))
            results = train_candidates(train, holdout, data_window=f"{months[0]}..{months[-1]}")
            version = register_best(results, f"Retrained on {months[0]}..{months[-1]} ({decision['reasons']})")
            passed, gate = quality_gate(version, holdout)
            promote_or_reject(version, passed, gate)
            mlflow.log_dict(gate.to_dict(orient="records"), "quality_gate.json")
            summary.update(new_version=version, gate="passed" if passed else "failed")

        # 6. Batch inference with whatever is champion now
        predictions = score_batch(score_month)
        summary.update(scored_month=score_month, scored_with_version=predictions["model_version"].iloc[0])
    return summary

Run the pipeline for **May** (April's labels arrive) and **June** (May's labels arrive, the month `SpeedyX` and storm
season started). Watch the log: in June, validation warns about a new carrier, monitoring raises the alarm, the
pipeline retrains, and the new challenger has to beat the champion **on the new data** to be promoted.

In [15]:
t0 = time.perf_counter()
history = [run_monthly_pipeline("2026-05-01"), run_monthly_pipeline("2026-06-01")]
log.info("two monthly runs took %.0f s", time.perf_counter() - t0)
pd.DataFrame(history)

2026-10-21 18:31:41,979 | INFO | [2026-04] validation passed (3000 rows)


2026-10-21 18:31:42,202 | WARNING | [2026-05 (to score)] carrier: new categories ['SpeedyX'] (25% of rows)


2026-10-21 18:31:42,203 | INFO | [2026-05 (to score)] validation passed (3000 rows)


2026-10-21 18:31:42,270 | INFO | scored 3000 orders of 2026-05 with late-delivery-risk v1 (600 to contact)


2026-10-21 18:31:42,409 | WARNING | [2026-05] carrier: new categories ['SpeedyX'] (25% of rows)


...

2026-10-21 18:31:47,204 | WARNING | [2026-06 (to score)] carrier: new categories ['SpeedyX'] (24% of rows)


2026-10-21 18:31:47,206 | INFO | [2026-06 (to score)] validation passed (3000 rows)


2026-10-21 18:31:47,289 | INFO | scored 3000 orders of 2026-06 with late-delivery-risk v3 (600 to contact)


2026-10-21 18:31:47,315 | INFO | two monthly runs took 5 s


,run_date,live_auc,monitoring,scored_month,scored_with_version,new_version,gate
0,2026-05-01,0.686,all green,2026-05,1,NaN,NaN
1,2026-06-01,0.696,PSI(weather_severity)=0.72; PSI(carrier)=2.04;...,2026-06,3,3.0,passed


In [16]:
# What exactly did monitoring see in June (May data)? Read it back from the MLflow parent run.
june_run = mlflow.search_runs(experiment_names=[CFG.experiment],
                              filter_string="tags.run_type = 'pipeline' and tags.run_date = '2026-06-01'")
cols = [c for c in june_run.columns if c.startswith(("metrics.psi_", "metrics.live_"))] + ["tags.retrain_reason"]
print(june_run[cols].T.to_string(header=False))

# Head-to-head on the May holdout, read from the gate report stored in the registry
new_mv = get_version_by_alias("champion")
print(f"\nGate report of version {new_mv.version}:")
print(pd.DataFrame(json.loads(new_mv.tags["gate_report"])).to_string(index=False))

metrics.live_auc                                                                                                  0.696
metrics.psi_weather_severity                                                                                      0.722
metrics.psi_distance_km                                                                                           0.002
metrics.live_mean_predicted                                                                                       0.366
metrics.live_recall_at_budget                                                                                       0.3
metrics.live_late_rate                                                                                            0.477
metrics.psi_n_items                                                                                               0.002
metrics.psi_carrier                                                                                               2.045
tags.retrain_reason            PSI(weath

This is the loop working end to end, without a human:
- **Validation** let the May data through but warned about the new carrier `SpeedyX`.
- **Monitoring** gives an important lesson: the old model's **live ROC-AUC did not drop** (about 0.70, even a bit
  above its holdout value). Watching only the ranking metric, everything would look fine! But the other two signals
  are loud: major drift (`carrier` and `weather_severity` PSI far above 0.25) and a **calibration gap**: the model
  predicted a late rate of about 37% while the real one was about 48%. It under-estimates `SpeedyX` delays because it
  has never seen that carrier (the one-hot encoder maps it to "no carrier information").
- The pipeline **retrained** on March-May, and the new challenger **beat the old champion by a wide margin on the
  same May holdout** (ROC-AUC ~0.79 vs ~0.69), passed the behavioural and latency checks, and was **promoted**.
  June orders were scored by the new version.

Let us check the business effect on June once its labels "arrive": compare the old and the new champion on the
same June orders.

In [17]:
june_labelled = ingest("2026-06", labelled=True)
old_model = load_version(get_version_by_alias("previous_champion").version)
new_model = load_version(get_version_by_alias("champion").version)
comparison = pd.DataFrame({
    f"old champion (v{get_version_by_alias('previous_champion').version})": evaluate(old_model, june_labelled),
    f"new champion (v{get_version_by_alias('champion').version})": evaluate(new_model, june_labelled),
}).T.round(3)
comparison

,roc_auc,pr_auc,recall_at_budget,usd_saved_per_month
old champion (v1),0.680,0.642,0.302,3344.0
new champion (v3),0.773,0.750,0.360,3976.0


On June orders that neither model has seen, the retrained model ranks much better (ROC-AUC ~0.77 vs ~0.68) and
catches more late parcels within the same 20% contact budget (~36% vs ~30% of all late parcels), which is worth
roughly **$600 more per month** at this hub (`usd_saved_per_month`). That dollar number, not the AUC, is what you
report to the operations manager.

---
## 11. Registry history, lineage and rollback

The registry now tells the whole story: which versions exist, which passed or failed the gate, and which aliases
point where. If the new champion misbehaves in production (a bug that the gate missed), **rollback** is a single
alias move back to `previous_champion`: the next scoring run picks it up, nothing is redeployed.

In [18]:
versions = client.search_model_versions(f"name = '{CFG.model_name}'")
aliases = {}
for alias in ("champion", "previous_champion", "challenger"):
    mv = get_version_by_alias(alias)
    if mv is not None:
        aliases.setdefault(mv.version, []).append(alias)

registry_table = pd.DataFrame([{
    "version": mv.version,
    "algorithm": mv.tags.get("algorithm"),
    "holdout_roc_auc": mv.tags.get("holdout_roc_auc"),
    "quality_gate": mv.tags.get("quality_gate"),
    "aliases": ", ".join(aliases.get(mv.version, [])),
    "trained_on": client.get_run(mv.run_id).data.tags.get("data_window"),
    "description": mv.description,
} for mv in versions]).sort_values("version")
registry_table

,version,algorithm,holdout_roc_auc,quality_gate,aliases,trained_on,description
2,1,logistic_regression,0.6714,passed,previous_champion,2026-01..2026-03,"First model, trained on 2026-01..2026-03"
1,2,gradient_boosting,0.6259,failed,,2026-01..2026-03,Teammate's refactor branch
0,3,logistic_regression,0.7856,passed,"champion, challenger",2026-03..2026-05,Retrained on 2026-03..2026-05 (PSI(weather_sev...


In [19]:
def rollback() -> None:
    """Emergency button: point 'champion' back to 'previous_champion' (and remember the bad version)."""
    current, previous = get_version_by_alias("champion"), get_version_by_alias("previous_champion")
    client.set_registered_model_alias(CFG.model_name, "champion", previous.version)
    client.set_registered_model_alias(CFG.model_name, "rolled_back", current.version)
    log.warning("ROLLBACK: champion v%s -> v%s", current.version, previous.version)


bad_version = get_version_by_alias("champion").version
rollback()
print("after rollback, production would load version", get_version_by_alias("champion").version)

# The incident is resolved, roll forward again (in real life: after a fix and a new gate run)
client.set_registered_model_alias(CFG.model_name, "champion", bad_version)
client.delete_registered_model_alias(CFG.model_name, "rolled_back")
print("rolled forward again, champion is version", get_version_by_alias("champion").version)

2026-10-21 18:31:47,887 | WARNING | ROLLBACK: champion v3 -> v1


after rollback, production would load version 1
rolled forward again, champion is version 3


---
## 12. From this notebook to Airflow / Prefect

Our `run_monthly_pipeline` is already structured like an orchestrated workflow. An orchestrator adds what a
notebook cannot: a **schedule** (cron `0 6 1 * *` = 06:00 on the 1st of each month), **retries** for flaky steps,
**alerts**, a UI with the history of every run, **back-fills** for past dates, and running steps on different machines.

| Our function | Airflow (`@task` in a `@dag`) | Prefect (`@task` in a `@flow`) |
|---|---|---|
| `ingest` + `require_valid` | `validate_data` task (fails the DAG run on error) | `@task(retries=2)` |
| `monitor` | `check_drift` task + `@task.branch` | plain `if` in the flow |
| `train_candidates` | `train` task (often on a bigger worker / Kubernetes pod) | `@task` |
| `register_best`, `quality_gate`, `promote_or_reject` | `register`, `gate`, `promote` tasks | `@task`s |
| `score_batch` | `batch_score` task | `@task` |
| `run_monthly_pipeline("2026-06-01")` | DAG run for logical date 2026-06-01 | `flow(run_date=...)` with a cron deployment |

A Prefect version is almost the same code (not executed here; see `03_pipeline_orchestration/01_orchestration.ipynb`):

```python
from prefect import flow, task

@task(retries=2, retry_delay_seconds=60)
def validate(month: str) -> pd.DataFrame:
    df = ingest(month, labelled=True)
    require_valid(df, labelled=True, batch_name=month)
    return df

@flow(name="late-delivery-monthly")
def monthly(run_date: str):
    labelled = validate(previous_month(run_date))
    decision = monitor_task(...)
    if decision["retrain"]:
        version = train_and_register(...)
        gate_and_promote(version)
    score_batch(...)

# monthly.serve(name="monthly", cron="0 6 1 * *")
```

And in Airflow (TaskFlow API):

```python
from airflow.decorators import dag, task
import pendulum

@dag(schedule="0 6 1 * *", start_date=pendulum.datetime(2026, 1, 1), catchup=False)
def late_delivery_monthly():
    @task
    def validate(ds=None): ...          # ds = the logical date Airflow passes in
    @task.branch
    def needs_retraining(report): return "train" if report["retrain"] else "batch_score"
    ...
late_delivery_monthly()
```

In CI/CD (see `04_ci_cd_for_ml/` and `../20_Git_and_DevOps/03_github_actions_cicd.ipynb`), the **quality gate
tests** also run on every pull request that changes training code, so the teammate's buggy refactor from
Section 7 would have been blocked *before* merging.

---
## 13. Automated tests for the pipeline

In a real repository these would be `pytest` tests in `tests/test_pipeline.py`, run in CI. They test the
**infrastructure**, independently of model quality (Rule #5 of Google's *Rules of ML*).

In [20]:
def test_validation_blocks_bad_batch():
    bad = ingest("2026-03", labelled=True).assign(n_items=-1)
    try:
        require_valid(bad, labelled=True, batch_name="test")
    except DataValidationError:
        return
    raise AssertionError("bad batch was not blocked")


def test_champion_passed_the_gate():
    assert get_version_by_alias("champion").tags["quality_gate"] == "passed"


def test_rejected_model_never_became_champion():
    assert get_version_by_alias("champion").version != v_buggy


def test_predictions_carry_model_version():
    preds = pd.read_parquet(PRED_DIR / "2026-06.parquet")
    assert preds["model_version"].nunique() == 1
    assert preds["model_version"].iloc[0] == get_version_by_alias("champion").version


def test_batch_scoring_is_idempotent():
    first = pd.read_parquet(PRED_DIR / "2026-06.parquet")["risk_of_late"]
    again = score_batch("2026-06")["risk_of_late"]                     # re-run the same month
    pd.testing.assert_series_equal(first, again)


def test_nothing_written_outside_work_dir():
    assert Path(mlflow.get_tracking_uri().replace("sqlite:///", "")).is_relative_to(WORK_DIR)
    assert mlflow.get_experiment_by_name(CFG.experiment).artifact_location.startswith(WORK_DIR.as_uri())


for test in [test_validation_blocks_bad_batch, test_champion_passed_the_gate, test_rejected_model_never_became_champion,
             test_predictions_carry_model_version, test_batch_scoring_is_idempotent, test_nothing_written_outside_work_dir]:
    test()
    print(f"PASSED  {test.__name__}")

PASSED  test_validation_blocks_bad_batch
PASSED  test_champion_passed_the_gate


2026-10-21 18:31:48,189 | WARNING | [2026-06 (to score)] carrier: new categories ['SpeedyX'] (24% of rows)


2026-10-21 18:31:48,191 | INFO | [2026-06 (to score)] validation passed (3000 rows)


2026-10-21 18:31:48,263 | INFO | scored 3000 orders of 2026-06 with late-delivery-risk v3 (600 to contact)


PASSED  test_rejected_model_never_became_champion
PASSED  test_predictions_carry_model_version
PASSED  test_batch_scoring_is_idempotent
PASSED  test_nothing_written_outside_work_dir


---
## 14. Summary, exercises and resources

| Practice | What we did | Where it is used in industry |
|---|---|---|
| Central config | all thresholds in a frozen `Config` | every production codebase |
| Data validation | contract with errors (stop) and warnings (log) | Great Expectations, pandera, TFDV, dbt tests |
| Time-based holdout | newest 14 days | any model that predicts the future |
| Experiment tracking | parent + nested runs, params, metrics, dataset, figures, model + signature | MLflow, W&B, Neptune |
| Run comparison | `mlflow.search_runs` leaderboard | model selection, reports |
| Model registry + aliases | `challenger` / `champion` / `previous_champion` | MLflow, SageMaker, Vertex AI, Databricks UC |
| Quality gate | head-to-head vs champion, business metric, behavioural + latency tests | CI/CD for ML, promotion policies |
| Batch inference | load `@champion`, predictions with model version | nightly/monthly scoring jobs |
| Monitoring | PSI drift + live AUC from joined labels | Evidently, WhyLabs, Arize, in-house dashboards |
| Retraining trigger | rule on drift and performance | continuous training (CT) |
| Orchestration | one idempotent function per run, mapped to Airflow / Prefect tasks | Airflow, Prefect, Dagster, Kubeflow |
| Rollback | move an alias back | incident response |

### Practice exercises
1. Change `min_auc_gain` to `0.05`. Does the June challenger still get promoted? What are the risks of a gate that is too strict or too loose?
2. Add a **fairness / slice check** to the gate: the challenger's recall must not drop for any single `warehouse`.
3. Add a third pipeline run for `2026-07-01`. Does monitoring stay green with the new champion? (Hint: think about
   which months are in its reference data.)
4. Replace the hand-written PSI with Evidently's `DataDriftPreset` (see `06_monitoring/01_monitoring.ipynb`).
5. Turn `run_monthly_pipeline` into a Prefect flow and run it locally with `monthly(run_date="2026-05-01")`.
6. Start `mlflow ui --backend-store-uri sqlite:///<WORK_DIR>/mlflow.db` (the path is printed in Section 1) and explore the runs.

### Resources
- MLflow model registry and aliases: https://mlflow.org/docs/latest/ml/model-registry/
- MLflow tracking (backend and artifact stores): https://mlflow.org/docs/latest/ml/tracking/
- Made With ML (MLOps course: testing, versioning, CI/CD, monitoring): https://madewithml.com/
- Google, *Rules of Machine Learning*: https://developers.google.com/machine-learning/guides/rules-of-ml
- Google Cloud, *MLOps: Continuous delivery and automation pipelines in ML*: https://cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning
- Breck et al., *The ML Test Score* (Google, 2017): https://research.google/pubs/the-ml-test-score-a-rubric-for-ml-production-readiness-and-technical-debt-reduction/
- Chip Huyen, *Designing Machine Learning Systems* (O'Reilly, 2022), chapters 8-9 (drift, continual learning).
- DataTalksClub MLOps Zoomcamp (free course, MLflow + Prefect + monitoring): https://github.com/DataTalksClub/mlops-zoomcamp
- Prefect docs: https://docs.prefect.io/ | Airflow TaskFlow tutorial: https://airflow.apache.org/docs/apache-airflow/stable/tutorial/taskflow.html

### Related notebooks in this repo
- Experiment tracking basics: [`01_experiment_tracking/01_mlflow.ipynb`](01_experiment_tracking/01_mlflow.ipynb)
- Model registry tools: [`05_model_registry/01_model_registry.ipynb`](05_model_registry/01_model_registry.ipynb)
- Monitoring tools: [`06_monitoring/01_monitoring.ipynb`](06_monitoring/01_monitoring.ipynb)
- Orchestration with Prefect / Airflow: [`03_pipeline_orchestration/01_orchestration.ipynb`](03_pipeline_orchestration/01_orchestration.ipynb)
- CI/CD for ML: [`04_ci_cd_for_ml/01_cicd_ml.ipynb`](04_ci_cd_for_ml/01_cicd_ml.ipynb)
- Testing ML code: [`07_testing_in_ml/01_testing_ml.ipynb`](07_testing_in_ml/01_testing_ml.ipynb)
- Data contracts: [`14_data_contracts.ipynb`](14_data_contracts.ipynb)
- Point-in-time feature SQL: [`../13_Databases/01_sql/02_industry_sql_for_ml_feature_extraction.ipynb`](../13_Databases/01_sql/02_industry_sql_for_ml_feature_extraction.ipynb)
- Team Git workflow for ML: [`../20_Git_and_DevOps/05_industry_team_git_workflow_for_ml.ipynb`](../20_Git_and_DevOps/05_industry_team_git_workflow_for_ml.ipynb)